# 🧠 LasmoidV1 — Production Model Pre-Training on Kaggle T4 GPU x2

**Developer:** theory903 (Abhishek Jha)  
**Architecture:** MLA + mHC + MoE + CQRS + MTP (Neuro-Symbolic MoE)  
**Tokenizer:** GPT-2 (vocab size: 50,257)  
**Context Window:** up to 1024 tokens  
**Weights & Optimizers:** Nesterov Muon for 2D weight matrices & AdamW for embed/bias parameters  
**SOTA Features:** EMA weight averaging, gradient checkpointing, W&B tracking, 8-stream curriculum, WSD schedule  
**Checkpoints:** Auto-synced to Hugging Face Hub (saves your progress across Kaggle's 12-hour session timeout limits)

### Supported Configurations
- **10M size**: `dim=128`, `n_layers=4`, `n_heads=4` — ~9.4M parameters
- **100M size**: `dim=512`, `n_layers=8`, `n_heads=8` — ~100.7M parameters
- **300M size**: `dim=768`, `n_layers=13`, `n_heads=12` — ~303.0M parameters (classic GPT-medium width with depth-first MoE)

### Prerequisites
Add `HF_TOKEN` as a Kaggle Secret (Settings → Secrets → Add New Secret).  
Name it exactly: **`HF_TOKEN`**  
For W&B tracking: `WANDB_API_KEY` as a Kaggle Secret (optional)

In [ ]:
# ── CELL 1: Environment diagnostics & internet check ──
import subprocess, sys, os, socket

print('=== GPU Info ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found!')

print('=== Python ===')
print(sys.version)

print('=== Disk Space ===')
subprocess.run(['df', '-h', '/kaggle/working'], text=True)

print('=== RAM ===')
with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line or 'MemAvailable' in line:
            print(line.strip())

print('\n=== Internet Access ===')
try:
    socket.create_connection(("8.8.8.8", 53), timeout=3)
    print("Internet is enabled ✓")
except OSError:
    print("⚠️ ERROR: Internet is disabled! Please toggle 'Internet' to ON in the right sidebar (under Settings) to enable downloads.")

In [ ]:
# ── CELL 2: Install dependencies ──
# Install core deps + optional W&B for experiment tracking
!pip install -q tiktoken datasets transformers huggingface_hub accelerate wandb
print('All dependencies installed ✓')

In [ ]:
# ── CELL 3: Clone/Pull repo from GitHub ──
import os

REPO_URL = 'https://github.com/Theory903/Lasmoid-V1.git'
WORK_DIR = '/kaggle/working/Lasmoid-V1'

if not os.path.exists(WORK_DIR):
    result = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    if result != 0:
        print('Git clone failed. You may need to enable Internet or upload files manually.')
    else:
        print(f'Cloned repo to {WORK_DIR} ✓')
else:
    print(f'Repo already exists at {WORK_DIR}. Force syncing with origin/main...')
    os.system(f'cd {WORK_DIR} && git fetch --all && git reset --hard origin/main')

print('\n=== Current Git Commit ===')
os.system(f'cd {WORK_DIR} && git log -n 1 --oneline')

In [ ]:
# ── CELL 3b: ALTERNATIVE — Upload files manually ──
# If you don't have a public GitHub repo, use Kaggle's "Add Input" → "Upload"
# to add a ZIP of your project, then unpack it.
#
# import shutil
# shutil.unpack_archive('/kaggle/input/lasmoid-v1/Lasmoid-V1.zip', '/kaggle/working/Lasmoid-V1')
print('If you used git clone in Cell 3, skip this cell.')

In [ ]:
# ── CELL 4: Hugging Face + W&B Authentication via Kaggle Secrets ──
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

secrets = UserSecretsClient()

# ── HF Hub (required) ──
try:
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError("Secret HF_TOKEN is empty")
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi(token=HF_TOKEN)
    username = api.whoami()['name']
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f'Logged in as: {username} ✓')
    print(f'Checkpoints will sync to: {username}/lasmoid-<model_size>')
except Exception as e:
    print('⚠️ ERROR: HF_TOKEN is not configured or failed to authenticate.')
    print('Please add your HF Write token as a secret named "HF_TOKEN" in Kaggle Settings -> Secrets.')

# ── W&B (optional) ──
try:
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    print('W&B: authenticated ✓')
except Exception:
    print('W&B: no WANDB_API_KEY secret found (tracking disabled)')

In [ ]:
# ── CELL 5: Verify model initialization ──
import sys, os
WORK_DIR = '/kaggle/working/Lasmoid-V1'
REPO_URL = 'https://github.com/Theory903/Lasmoid-V1.git'

if not os.path.exists(f'{WORK_DIR}/train_kaggle.py'):
    print('Repository files missing. Resetting and cloning...')
    os.system(f'rm -rf {WORK_DIR}')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')

sys.path.insert(0, WORK_DIR)

import torch
from train_kaggle import MODEL_CONFIGS
from inference.model import LasmoidV1, ModelArgs

# Select model size: "10M", "100M" or "300M"
MODEL_SIZE = "10M"

cfg = MODEL_CONFIGS[MODEL_SIZE].copy()
cfg.pop("_verified_params", None)
cfg["max_seq_len"] = 1024
cfg["max_batch_size"] = 4

model_args = ModelArgs(**cfg)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LasmoidV1(model_args).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Device: {device}')
print(f'Model: {MODEL_SIZE} | LasmoidV1, {total_params:,} params')
print(f'Developer: theory903 (Abhishek Jha)')

# Quick forward pass sanity check
x = torch.randint(0, 50257, (2, 64)).to(device)
with torch.no_grad():
    out = model(x, x)
print(f'Forward pass output shape: {out[0].shape}')
print('Model initialized successfully ✓')

# ── Gradient checkpointing test (optional memory saver) ──
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    with torch.no_grad():
        out2 = model(x, x)
    print(f'Gradient checkpointing: OK, output shape {out2[0].shape}')

In [ ]:
# ── CELL 6: LAUNCH SFT TRAINING ──
import os, sys

WORK_DIR = '/kaggle/working/Lasmoid-V1'
print('Syncing code to latest commit on main...')
os.system(f'cd {WORK_DIR} && git fetch --all && git reset --hard origin/main')

cmd = [
    'torchrun', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '10M',
    '--max_iters',     '1000',
    '--batch_size',    '4',
    '--grad_accum',    '8',
    '--lr',            '3e-4',
    '--muon_lr',       '2e-3',
    '--warmup_steps',  '100',
    '--decay_frac',    '0.5',
    '--lr_min_ratio',  '0.1',
    '--weight_decay',  '0.1',
    '--grad_clip',     '1.0',
    '--dtype',         'bf16',
    '--gradient_checkpointing',
    '--ema_decay',     '0.9999',
    '--wandb',         'lasmoid-v1',
    '--phase',         'sft',
    '--save_interval', '200',
    '--log_interval',  '10',
    '--session_hours', '2.0',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints',
]

print('Starting LasmoidV1 SFT Phase training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 70)

# Use os.system to avoid PyTorch/CUDA fork-exec SIGABRT issues
cmd_str = ' '.join(cmd)
ret = os.system(f'cd {WORK_DIR} && {cmd_str}')
exit_code = os.waitstatus_to_exitcode(ret)
print(f'\nTraining finished with return code: {exit_code}')

In [ ]:
# ── CELL 6b: LAUNCH RL PHASE (Parallel-R1) ──
import os, sys

WORK_DIR = '/kaggle/working/Lasmoid-V1'
print('Syncing code to latest commit on main...')
os.system(f'cd {WORK_DIR} && git fetch --all && git reset --hard origin/main')

cmd = [
    'torchrun', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '10M',
    '--max_iters',     '1000',
    '--batch_size',    '4',
    '--grad_accum',    '8',
    '--lr',            '5e-6',
    '--muon_lr',       '1e-4',
    '--warmup_steps',  '100',
    '--decay_frac',    '0.5',
    '--lr_min_ratio',  '0.1',
    '--weight_decay',  '0.1',
    '--grad_clip',     '1.0',
    '--dtype',         'bf16',
    '--gradient_checkpointing',
    '--ema_decay',     '0.9999',
    '--wandb',         'lasmoid-v1',
    '--phase',         'rl',
    '--save_interval', '200',
    '--log_interval',  '10',
    '--session_hours', '2.0',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints',
]

print('Starting LasmoidV1 RL Phase (Parallel-R1) training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 70)

# Use os.system to avoid PyTorch/CUDA fork-exec SIGABRT issues
cmd_str = ' '.join(cmd)
ret = os.system(f'cd {WORK_DIR} && {cmd_str}')
exit_code = os.waitstatus_to_exitcode(ret)
print(f'\nTraining finished with return code: {exit_code}')

In [ ]:
# ── CELL 6c: LAUNCH FST PHASE (Fast-Slow Training) ──
import os, sys

WORK_DIR = '/kaggle/working/Lasmoid-V1'
print('Syncing code to latest commit on main...')
os.system(f'cd {WORK_DIR} && git fetch --all && git reset --hard origin/main')

cmd = [
    'torchrun', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '10M',
    '--max_iters',     '1000',
    '--batch_size',    '4',
    '--grad_accum',    '8',
    '--lr',            '1e-5',
    '--muon_lr',       '2e-4',
    '--warmup_steps',  '100',
    '--decay_frac',    '0.5',
    '--lr_min_ratio',  '0.1',
    '--weight_decay',  '0.1',
    '--grad_clip',     '1.0',
    '--dtype',         'bf16',
    '--gradient_checkpointing',
    '--ema_decay',     '0.9999',
    '--wandb',         'lasmoid-v1',
    '--phase',         'fst',
    '--save_interval', '200',
    '--log_interval',  '10',
    '--session_hours', '2.0',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints',
]

print('Starting LasmoidV1 FST Phase (GEPA) training...')
print(f'Command: {" ".join(cmd)}')
print('=' * 70)

# Use os.system to avoid PyTorch/CUDA fork-exec SIGABRT issues
cmd_str = ' '.join(cmd)
ret = os.system(f'cd {WORK_DIR} && {cmd_str}')
exit_code = os.waitstatus_to_exitcode(ret)
print(f'\nTraining finished with return code: {exit_code}')

In [ ]:
# ── CELL 7: Verify checkpoint on HF Hub ──
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import os

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    api = HfApi()
    username = api.whoami(token=HF_TOKEN)['name']
    
    # Change this to match the model size you trained: "100M" or "300M"
    MODEL_SIZE = "10M"
    repo_id = f'{username}/lasmoid-{MODEL_SIZE.lower()}'

    files = list(api.list_repo_files(repo_id=repo_id, token=HF_TOKEN))
    pt_files = [f for f in files if f.endswith('.pt')]
    csv_files = [f for f in files if f.endswith('.csv')]
    print(f'Repository: {repo_id}')
    print(f'Checkpoints found Remotely ({len(pt_files)}):')
    for f in sorted(pt_files):
        print(f'  - {f}')
    if csv_files:
        print(f'\nMetrics CSV: {csv_files[0]} (download for loss curve plot)')
    if not pt_files:
        print('No checkpoints yet — training may still be in progress.')
except Exception as e:
    print(f'Error listing checkpoints: {e}')

In [ ]:
# ── CELL 8: Quick generation test after training ──
import sys, torch, tiktoken, glob, os
WORK_DIR = '/kaggle/working/Lasmoid-V1'
sys.path.insert(0, WORK_DIR)

from train_kaggle import MODEL_CONFIGS
from inference.model import LasmoidV1, ModelArgs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = tiktoken.get_encoding('gpt2')

# Select model size to load: "10M", "100M" or "300M"
MODEL_SIZE = "10M"

cfg = MODEL_CONFIGS[MODEL_SIZE].copy()
cfg.pop("_verified_params", None)
cfg["max_seq_len"] = 1024
cfg["max_batch_size"] = 1

model_args = ModelArgs(**cfg)

# Load latest checkpoint
ckpts = sorted(glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_checkpoint_step_*.pt'))
if not ckpts:
    ckpts = glob.glob(f'{WORK_DIR}/checkpoints/lasmoid_latest.pt')

if ckpts:
    latest = ckpts[-1]
    print(f'Loading: {latest}')
    model = LasmoidV1(model_args).to(device)
    ckpt = torch.load(latest, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    
    # Strip compilation prefix if any
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('_orig_mod.'):
            clean_state_dict[k[10:]] = v
        else:
            clean_state_dict[k] = v
            
    model.load_state_dict(clean_state_dict)
    model.eval()

    prompt = 'The meaning of intelligence is'
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long).to(device)

    print(f'Prompt: {prompt}')
    print('Generation:', end=' ')
    with torch.no_grad():
        for _ in range(50):
            logits, _, _, _ = model(x, x)
            next_tok = logits[0, -1].argmax().item()
            print(tokenizer.decode([next_tok]), end='', flush=True)
            x = torch.cat([x, torch.tensor([[next_tok]]).to(device)], dim=1)
            if x.shape[1] >= model_args.max_seq_len:
                break
    print()
else:
    print('No checkpoint found. Run Cell 6 first.')

In [ ]:
# ── CELL 9: Quick smoke test (50 steps) ──
# Run this BEFORE launching production training to verify it end-to-end.
import os, sys, glob

WORK_DIR = '/kaggle/working/Lasmoid-V1'

# Always sync to the latest git commit on origin/main before running tests
print('Syncing workspace with remote GitHub repo...')
os.system(f'cd {WORK_DIR} && git fetch --all && git reset --hard origin/main')

print('\n=== Current Git Commit ===')
os.system(f'cd {WORK_DIR} && git log -n 1 --oneline')

# Step 1: Verify all 8 data streams open correctly
print('\n=== Step 1: --data_check (verify 8 streams) ===')
cmd_check = [
    sys.executable, f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '10M',
    '--data_check',
    '--device', 'cpu'
]

# Use os.system to completely avoid PyTorch/CUDA fork-exec SIGABRT issues
cmd_check_str = ' '.join(cmd_check)
ret_check = os.system(f'cd {WORK_DIR} && {cmd_check_str}')
exit_code_check = os.waitstatus_to_exitcode(ret_check)
if exit_code_check != 0:
    raise RuntimeError(f"Data check failed! Exit code: {exit_code_check}")
print('Data check passed!')

# Step 2: Train 50 steps with full pipeline (incl. EMA + grad ckpt + bf16)
print('\n=== Step 2: 50-step training smoke test ===')
cmd_train = [
    sys.executable, '-m', 'torch.distributed.run', '--nproc_per_node=2', f'{WORK_DIR}/train_kaggle.py',
    '--model_size', '10M',
    '--max_iters',      '50',
    '--batch_size',     '2',
    '--grad_accum',     '2',
    '--lr',             '3e-4',
    '--muon_lr',        '2e-3',
    '--warmup_steps',   '10',
    '--decay_frac',     '0.2',
    '--lr_min_ratio',   '0.1',
    '--weight_decay',   '0.1',
    '--grad_clip',      '1.0',
    '--dtype',          'bf16',
    '--gradient_checkpointing',
    '--ema_decay',      '0.9999',
    '--save_interval',  '25',
    '--log_interval',   '5',
    '--checkpoint_dir', f'{WORK_DIR}/checkpoints_test',
]

# Use os.system to completely avoid PyTorch/CUDA fork-exec SIGABRT issues
cmd_train_str = ' '.join(cmd_train)
ret_train = os.system(f'cd {WORK_DIR} && {cmd_train_str}')
exit_code_train = os.waitstatus_to_exitcode(ret_train)
if exit_code_train != 0:
    raise RuntimeError(f"Training smoke test failed! Exit code: {exit_code_train}")
print('Training smoke test passed!')

# Step 3: Verify artifacts exist
print('\n=== Step 3: Verify artifacts ===')
ckpts = sorted(glob.glob(f'{WORK_DIR}/checkpoints_test/*.pt'))
csvs = sorted(glob.glob(f'{WORK_DIR}/checkpoints_test/*.csv'))
print(f'Checkpoints: {len(ckpts)}')
print(f'Metrics CSVs: {len(csvs)}')
sizes = [os.path.getsize(f) / 1e6 for f in ckpts]
print(f'Checkpoint sizes (MB): {[round(s, 1) for s in sizes]}')

assert len(ckpts) >= 1, 'No checkpoint saved'
assert len(csvs) >= 1, 'No metrics CSV saved'
assert any(s > 100 for s in sizes) or any(s > 10 for s in sizes), f'Checkpoint sizes: {[round(s, 1) for s in sizes]}MB'
print('All artifacts verified!')

print('\n=== SMOKE TEST PASSED ✅ ===')
print('If all checks passed, proceed to Cell 6 for full training.')